# Vectorless RAG with PageIndex

This notebook builds a model-agnostic, vectorless RAG workflow over PDF files stored in a local `data/` folder.

The flow is:

1. Install and configure dependencies.
2. Load a PDF from `data/` and submit it to PageIndex.
3. Use an LLM to reason over the PageIndex tree and select relevant nodes.
4. Extract the evidence text from those nodes.
5. Ask the LLM for a final answer plus a concise explainability summary.

The notebook is intentionally modular so each step can be inspected or replaced independently.

## What is Vectorless RAG?

Traditional RAG works like tearing pages out of a book, cutting them into small
scraps, and tossing them into a hat. When you ask a question, the AI pulls out a few
random scraps and tries to piece together an answer. Important context gets lost
when text is chopped into tiny chunks.

Vectorless RAG takes a different approach. Instead of chopping the document up,
it builds a structured table of contents — a tree — that preserves the full layout
and hierarchy of the original document. When you ask a question, the AI looks at the
tree, picks the sections that matter, and pulls in the *whole*, unbroken text from
those sections.

The result: more accurate answers, better context, and no embeddings or vector
databases required.

The PageIndex service handles the tree-building step automatically when you submit
your PDF — no configuration needed on your end.

## 1. Prerequisites

This notebook assumes you have:

- A PDF document in a local `data/` folder.
- A PageIndex API key for tree generation.
- A Groq API key for the free LLM endpoint.

The LLM layer uses [Groq](https://groq.com/) via its OpenAI-compatible endpoint. You can choose between `llama-3.3-70b-versatile` (larger) and `llama-3.1-8b-instant` (faster) at runtime.

### API Key Setup

You will need two API keys:

| Key | Where to get it |
|-----|------------------|
| `GROQ_API_KEY` | [https://console.groq.com/keys](https://console.groq.com/keys) |
| `PAGEINDEX_API_KEY` | [https://www.pageindex.ai](https://www.pageindex.ai) |

Set them as environment variables or in a `.env` file in the same directory as this notebook:

```
GROQ_API_KEY=gsk_...
PAGEINDEX_API_KEY=...
```

In [ ]:
# Install core dependencies: pageindex (tree API), openai (LLM client),# python-dotenv (env file loading), pymupdf (PDF parsing), ipywidgets (UI widgets)%pip install -q --upgrade pageindex openai python-dotenv pymupdf ipywidgets

## 2. Configuration

The notebook reads environment variables from a `.env` file or your shell.
If either key is missing, you will be prompted to enter it securely.

- `GROQ_API_KEY`: Groq API key (from [console.groq.com/keys](https://console.groq.com/keys)).
- `PAGEINDEX_API_KEY`: PageIndex API key.
- `PDF_NAME`: Optional filename in `data/` if you want to pin a specific PDF.

You will be prompted to choose between two Groq models at runtime:
- `llama-3.3-70b-versatile` — larger, more capable.
- `llama-3.1-8b-instant` — faster, lighter.

Groq provides a free OpenAI-compatible endpoint, so no `LLM_BASE_URL` is needed.

In [1]:
# --- Imports ---
# Standard library: file paths (Path), secure input (getpass), JSON handling,# environment variables, regex, timing, and type hints.from pathlib import Path
import getpass
import json
import os
import re
import time
from typing import Any

# Third-party: dotenv for .env file loading, openai for the LLM client,# pageindex for the tree-building API and its utility functions.from dotenv import load_dotenv
from openai import AsyncOpenAI
from pageindex import PageIndexClient
import pageindex.utils as pi_utils

# Load API keys from a .env file if present, falling back to shell env vars.load_dotenv()

# --- Directory constants ---
DATA_DIR = Path("data")
CACHE_DIR = DATA_DIR / "cache"  # Stores pageindex tree JSON to avoid re-processingLLM_BASE_URL = "https://api.groq.com/openai/v1"  # Groq's OpenAI-compatible endpoint
# --- API key handling ---
# Try env vars first; if empty, prompt the user interactively so keys are never hardcoded.PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "").strip()
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = getpass.getpass("Enter your PAGEINDEX_API_KEY: ").strip()

LLM_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
if not LLM_API_KEY:
    LLM_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ").strip()

PDF_NAME = os.getenv("PDF_NAME")  # Optional: pin a specific PDF from the env
# --- Model selection UI ---
# Presents the two available Groq/Llama models and reads the user's choice.MODELS = {
    "1": ("llama-3.3-70b-versatile", "Llama 3.3 70B — larger, more capable"),
    "2": ("llama-3.1-8b-instant", "Llama 3.1 8B — faster, lighter"),
}

print("Choose an LLM model:")
for key, (name, desc) in MODELS.items():
    print(f"  {key}. {name} — {desc}")

choice = input("Enter 1 or 2: ").strip()
LLM_MODEL = MODELS.get(choice, MODELS["1"])[0]  # Default to model 1 on bad input
print(f"Using model: {LLM_MODEL}")

# --- Client initialization ---
# PageIndex client for tree building; OpenAI-compatible client for Groq LLM calls.pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY) if PAGEINDEX_API_KEY else None
llm_client = AsyncOpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL) if LLM_API_KEY else None


# --- Helper: extract_json ---
# LLMs sometimes wrap JSON in markdown fences or add prose. This extracts the# first JSON object from arbitrary text using regex, which is more robust than# assuming the LLM returns clean JSON every time.def extract_json(text: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"Expected JSON output, got: {text[:500]}")
    return json.loads(match.group(0))


# --- Helper: call_llm ---
# Wraps the OpenAI chat completion call with retry logic for rate limits.# Uses exponential backoff with jitter to spread retries and avoid thundering herd.async def call_llm(system_prompt: str, user_prompt: str, model: str = LLM_MODEL, temperature: float = 0.0) -> str:
    if llm_client is None:
        raise RuntimeError("GROQ_API_KEY is not configured.")
    import openai
    import asyncio
    import random
    max_attempts = 6
    for attempt in range(1, max_attempts + 1):
        try:
            response = await llm_client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            return response.choices[0].message.content.strip()
        except openai.RateLimitError:
            if attempt == max_attempts:
                raise RuntimeError("Groq service is busy, please try again shortly.")
            # Exponential backoff: 2^attempt seconds + random jitter, capped at 30s
            wait = min(2 ** attempt + random.uniform(0, 2), 30)
            print(f"Rate limited, retrying in {wait:.0f}s... (attempt {attempt}/{max_attempts})")
            await asyncio.sleep(wait)


# --- Helper: call_llm_and_parse ---
# Calls the LLM and parses JSON from the response. If the first attempt returns# malformed JSON, it sends a corrective follow-up asking the model to fix its output.# This "corrective retry" pattern handles the common case where models add extra text.async def call_llm_and_parse(system_prompt: str, user_prompt: str, model: str = LLM_MODEL, temperature: float = 0.0) -> dict[str, Any]:
    reply = await call_llm(system_prompt, user_prompt, model, temperature)
    try:
        return extract_json(reply)
    except ValueError:
        print("Received malformed JSON, sending one corrective retry...")
        reply2 = await call_llm(
            system_prompt,
            f"Your previous reply was not valid JSON:\n\n{reply}\n\nReturn ONLY the valid JSON object, with no extra text, no markdown fences, and no explanation.",
            model,
            temperature,
        )
        try:
            return extract_json(reply2)
        except ValueError:
            raise RuntimeError("The AI model failed to return valid JSON after 2 attempts. Please try rephrasing your question.")


# --- Helper: preview_text ---
# Truncates long text for display with an ellipsis, keeping output readable.def preview_text(text: str, limit: int = 1200) -> str:
    text = text.strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "..."

Choose an LLM model:
  1. llama-3.3-70b-versatile — Llama 3.3 70B — larger, more capable
  2. llama-3.1-8b-instant — Llama 3.1 8B — faster, lighter


Enter 1 or 2:  1


Using model: llama-3.3-70b-versatile


## 3. Load a PDF from `data/`

Upload a PDF using the button above, or place PDF files directly in the `data/` folder.
Then use the dropdown to select which PDF to process.

The PageIndex step below creates the tree structure for this document without using
embeddings or a vector database.

In [2]:
# --- File upload widget setup ---
import ipywidgets as widgets
from IPython.display import display

DATA_DIR.mkdir(exist_ok=True)

# Create a file upload button that only accepts PDF files.upload_output = widgets.Output()
upload_button = widgets.FileUpload(
    description='Upload PDF',
    accept='.pdf',
    multiple=False,
    style={'description_width': 'initial'},
)

display(upload_button)

# Track already-saved filenames to avoid writing duplicates on re-upload._saved_files = set()

# Upload handler: saves each uploaded file's bytes to the data/ directory.def on_upload_change(change):
    with upload_output:
        upload_output.clear_output()
        if change['new']:
            for entry in change['new']:
                if entry.name in _saved_files:
                    continue
                if not entry.content:
                    continue
                save_path = DATA_DIR / entry.name
                save_path.write_bytes(entry.content)
                _saved_files.add(entry.name)
                print(f"Saved: {save_path}")

upload_button.observe(on_upload_change, names='value')
display(upload_output)

# --- PDF selection dropdown ---
# List all PDFs in data/ and let the user pick one.pdf_files = sorted(p.name for p in DATA_DIR.glob('*.pdf'))
if not pdf_files:
    print("⚠ No PDF files found. Upload a PDF above, then re-run this cell.")
else:
    pdf_dropdown = widgets.Dropdown(
        options=pdf_files,
        value=pdf_files[0],
        description='Select PDF:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
    )

    display(pdf_dropdown)

    pdf_path = DATA_DIR / pdf_files[0]

    def on_pdf_change(change):
        global pdf_path
        pdf_path = DATA_DIR / change['new']

    pdf_dropdown.observe(on_pdf_change, names='value')

# --- PDF validation ---
# Verify the file starts with the %PDF magic bytes and is under 20 MB.MAX_PDF_SIZE_MB = 20
with open(pdf_path, 'rb') as f:
    header = f.read(4)
if header != b'%PDF':
    raise ValueError(f"{pdf_path.name} does not appear to be a valid PDF file (missing %PDF signature).")

size_mb = pdf_path.stat().st_size / (1024 * 1024)
if size_mb > MAX_PDF_SIZE_MB:
    raise ValueError(f"{pdf_path.name} is {size_mb:.1f} MB, which exceeds the {MAX_PDF_SIZE_MB} MB limit.")

print(f"Using PDF: {pdf_path}")

FileUpload(value=(), accept='.pdf', description='Upload PDF')

Output()

Dropdown(description='Select PDF:', layout=Layout(width='500px'), options=('synthetic_medicare_plus_policy_det…

Using PDF: data/synthetic_medicare_plus_policy_detailed.pdf


## 4. Build the PageIndex tree

This cell submits the selected PDF to PageIndex and retrieves the generated tree.

The tree is cached locally in `data/cache/` so subsequent runs skip the PageIndex API call for the same document.

If the document is still processing, the notebook polls with a progress bar until the tree is ready.

In [3]:
# --- Guard: ensure a PDF was selected in the previous cell ---    if 'pdf_path' not in dir():
    raise NameError(
        "'pdf_path' is not defined. Please run the PDF selection cell above first."
    )

# --- Cache check ---
# Avoid redundant PageIndex API calls by reusing a previously built tree.CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"{pdf_path.stem}_tree.json"

if cache_path.exists():
    print(f"Loading cached tree from {cache_path}")
    tree = json.loads(cache_path.read_text())
else:
    if pi_client is None:
        raise RuntimeError("PAGEINDEX_API_KEY is not configured.")

    # Submit the PDF to the PageIndex API for asynchronous tree construction.    submitted = pi_client.submit_document(str(pdf_path))
    doc_id = submitted.get("doc_id") or submitted.get("result", {}).get("doc_id")
    if not doc_id:
        raise RuntimeError(f"Could not read doc_id from PageIndex response: {submitted}")

    print(f"Submitted document id: {doc_id}")
    print("Waiting for PageIndex to process the document...")

    # --- Polling loop with spinner ---
# PageIndex processes documents asynchronously; poll every 5s until ready or timeout.    poll_interval = 5
    max_wait = 300
    elapsed = 0
    spinner = ["|", "/", "-", "\\"]
    while elapsed < max_wait:
        if pi_client.is_retrieval_ready(doc_id):
            break
        idx = (elapsed // poll_interval) % len(spinner)
        print(f"\r  {spinner[idx]}  Waiting... {elapsed}s / {max_wait}s", end="", flush=True)
        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        raise RuntimeError(
            f"PageIndex did not finish within {max_wait}s. Rerun this cell later."
        )
    print(f"\r  Done! Processed in {elapsed}s.{' ' * 20}")

    # --- Retrieve and cache the tree ---    tree_response = pi_client.get_tree(doc_id, node_summary=True)
    tree = tree_response.get("result", tree_response)

    cache_path.write_text(json.dumps(tree))
    print(f"Tree cached to {cache_path}")

print("Tree ready. Top-level preview:")
pi_utils.print_tree(tree[:2] if isinstance(tree, list) else tree)

Loading cached tree from data/cache/synthetic_medicare_plus_policy_detailed_tree.json
Tree ready. Top-level preview:
[{'title': 'POLICY WORDING, BENEFIT SCHEDULE AND UND...',
  'node_id': '0000',
  'summary': 'This document provides the introductory ...'},
 {'title': 'SECTION 3 - INSURING AGREEMENTS',
  'node_id': '0001',
  'summary': "This section outlines the insurer's obli..."}]


## 5. Prepare retrieval helpers

The next cells use the tree structure for two passes:

1. Tree search: the LLM selects the node IDs most likely to contain the answer.
2. Evidence extraction: the notebook pulls the full text for those nodes and passes it into the final answer prompt.

This keeps the workflow vectorless while still giving you transparent, inspectable retrieval decisions.

In [4]:
# --- Flatten the tree into a node_id -> node dict ---# create_node_mapping recursively walks the tree so we can look up any node by ID.node_map = pi_utils.create_node_mapping(tree)


# --- Prepare tree text for LLM prompts ---
# Strip the full text content from each node to keep the prompt compact;# the LLM only needs titles and summaries to decide which nodes to retrieve.def tree_as_prompt_text(document_tree: object) -> str:
    tree_copy = document_tree.copy() if isinstance(document_tree, dict) else document_tree
    tree_without_text = pi_utils.remove_fields(tree_copy, fields=["text"])
    return json.dumps(tree_without_text, indent=2)


# --- Collect evidence from selected nodes ---
# Given a list of node IDs, pull the full text from the node_map and tag each# passage with its source metadata (node ID, page number, title) for traceability.def collect_evidence(node_ids: list[str]) -> dict:
    """Return {'text': str, 'sources': [{'node_id': ..., 'page': ..., 'title': ...}]}"""
    parts: list[str] = []
    sources: list[dict] = []
    for node_id in node_ids:
        node = node_map.get(node_id)
        if not node:
            continue
        text = node.get("text", "")
        title = node.get("title", node_id)
        page_index = node.get("page_index", "?")
        parts.append(f"[node={node_id} page={page_index} title={title}]\n{text}")
        sources.append({"node_id": node_id, "page": page_index, "title": title})
    return {"text": "\n\n".join(parts), "sources": sources}


print(f"Indexed nodes: {len(node_map)}")

Indexed nodes: 24


## 6. Ask a question

This step runs the full vectorless RAG cycle for a single question, in three stages:

1. **Tree search:** the LLM receives the question and a compact PageIndex tree of node titles and summaries, then selects the node IDs most likely to contain the answer, along with a short retrieval explanation.
2. **Evidence extraction:** the selected node IDs are converted back into their source text. That evidence is the only context the final answer step is allowed to use.
3. **Final answer:** the LLM answers strictly from the retrieved evidence and returns a structured response with a `final_answer` and an `explainability` summary of which evidence was used.

All three stages run inside a single `answer_question()` call, so the result you see is always generated end-to-end from the current question — never a mix of outputs from different questions.

In [5]:
# --- Three-stage RAG pipeline ---
# Stage 1: LLM reasons over the tree to select relevant node IDs.# Stage 2: Full text is extracted from those nodes as evidence.# Stage 3: LLM generates a final answer grounded only in that evidence.async def answer_question(question: str, model: str = LLM_MODEL) -> dict[str, Any]:
    """Run retrieval -> evidence extraction -> final answer as one atomic step."""
    question = question.strip()
    if not question:
        raise ValueError("A question is required to continue.")

    # --- Step 1: reasoning-based tree search ---
# The LLM receives the compact tree (titles + summaries only) and picks# which node IDs are most likely to contain the answer.    retrieval_system_prompt = """
You are a document retrieval assistant for a vectorless RAG system.

You will receive a user question and a PageIndex tree made of node titles and summaries.
Select only the node IDs that are likely to contain answer-bearing evidence.

Return valid JSON with this shape:
{
  "thinking": "short, evidence-based retrieval explanation",
  "node_list": ["node_id_1", "node_id_2"]
}

Do not output markdown, prose, or extra keys.
Keep the explanation brief and grounded in the tree.
""".strip()

    retrieval_user_prompt = f"""
Question:
{question}

PageIndex tree:
{tree_as_prompt_text(tree)}
""".strip()

    retrieval_json = await call_llm_and_parse(retrieval_system_prompt, retrieval_user_prompt, model=model)
    selected_node_ids = retrieval_json.get("node_list", [])
    retrieval_thinking = retrieval_json.get("thinking", "")

    # --- Retry with broader search if nothing was found ---
# Some questions need looser matching; this second pass tells the LLM to err on inclusion.    if not selected_node_ids:
        print("No relevant sections found on first attempt, retrying with a broader search...")
        retry_retrieval_user_prompt = f"""
Your previous search returned no relevant sections for this question:

{question}

Look again more carefully. Include any section that is even loosely or 
indirectly relevant — err on the side of including a section rather than 
excluding it, unless you are certain nothing in the document relates to 
this question.

PageIndex tree:
{tree_as_prompt_text(tree)}
""".strip()
        retry_json = await call_llm_and_parse(retrieval_system_prompt, retry_retrieval_user_prompt, model=model)
        selected_node_ids = retry_json.get("node_list", [])
        retrieval_thinking = retry_json.get("thinking", "")

    # If still no nodes matched, return early with a clear message.    if not selected_node_ids:
        return {
            "question": question,
            "selected_node_ids": [],
            "retrieval_thinking": retrieval_thinking,
            "evidence_text": "",
            "final_answer": "No relevant section was found in the document for this question.",
            "explainability": {},
        }

    # --- Step 2: evidence extraction ---
# Pull the full unbroken text from each selected node, tagged with page numbers.    evidence = collect_evidence(selected_node_ids)
    evidence_text = evidence["text"]
    evidence_sources = evidence["sources"]
    source_pages = sorted(set(s["page"] for s in evidence_sources if s["page"] != "?"))

    # --- Step 3: final answer generation ---
# The LLM answers strictly from the retrieved evidence.    final_system_prompt = """
You are a careful assistant answering questions about a PDF document.

Use only the evidence text provided by the notebook.
Do not invent facts or rely on outside knowledge.
If the evidence does not contain the answer, say so explicitly instead of guessing.

Each evidence passage is tagged with its source page number.
Use these page numbers to cite your sources in the answer.

Return valid JSON with this shape:
{
  "final_answer": "concise answer for the user",
  "source_pages": [1, 3, 5],
  "explainability": {
    "retrieval_summary": "brief note about why the selected evidence is relevant",
    "evidence_used": ["short evidence note 1", "short evidence note 2"]
  }
}

Keep the explanation short and grounded in the provided context.
""".strip()

    evidence_for_prompt = evidence_text.strip() or "(No evidence text was retrieved for this question.)"
    # Cap evidence length to prevent context window overflow and keep the LLM focused.    max_evidence_chars = 8000
    if len(evidence_for_prompt) > max_evidence_chars:
        evidence_for_prompt = evidence_for_prompt[:max_evidence_chars].rstrip() + "\n\n... (evidence truncated for brevity)"
    final_user_prompt = f"""
Question:
{question}

Evidence context:
{evidence_for_prompt}

Available source pages: {source_pages}
""".strip()

    final_json = await call_llm_and_parse(final_system_prompt, final_user_prompt, model=model)

    return {
        "question": question,
        "selected_node_ids": selected_node_ids,
        "retrieval_thinking": retrieval_thinking,
        "evidence_text": evidence_text,
        "evidence_sources": evidence_sources,
        "source_pages": final_json.get("source_pages", source_pages),
        "final_answer": final_json.get("final_answer", ""),
        "explainability": final_json.get("explainability", {}),
    }

In [ ]:
# Prompt the user for a question and run the full RAG pipeline.QUESTION = input("Enter your question about the PDF: ").strip()

result = await answer_question(QUESTION)

### 6a. Question & Retrieval Explainability

In [ ]:
# Display the original question and the LLM's retrieval reasoning.print("=" * 70)
print("QUESTION")
print("=" * 70)
print(result["question"])

print()
print("=" * 70)
print("RETRIEVAL EXPLAINABILITY")
print("=" * 70)
print("Selected node IDs:", result["selected_node_ids"])
print(result["retrieval_thinking"])

### 6b. Retrieved Evidence

In [ ]:
# Show which nodes and pages contributed evidence to the answer.if result.get("evidence_sources"):
    print("Source pages:", result.get("source_pages", []))
    print()
    for src in result["evidence_sources"]:
        print(f"  node={src['node_id']}  page={src['page']}  {src['title']}")
else:
    print("(No evidence retrieved.)")

In [ ]:
# Show a truncated preview of the raw evidence text passed to the final answer step.print("=" * 70)
print("EVIDENCE PREVIEW")
print("=" * 70)
print(preview_text(result["evidence_text"], limit=3000))

### 6c. Final Answer

In [ ]:
# Display the LLM's final answer and its cited source pages.print("=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(result["final_answer"])

print()
print("Source pages:", result.get("source_pages", []))

### 6d. Explainability

In [ ]:
# Display the structured explainability JSON from the final answer step.print("=" * 70)
print("EXPLAINABILITY")
print("=" * 70)
print(json.dumps(result["explainability"], indent=2))

### 6e. Groundedness Check

In [ ]:
# --- Groundedness check ---
# Uses a separate LLM call to verify that the final answer's claims# are actually supported by the retrieved evidence.if result["evidence_text"]:
    groundedness_system_prompt = """
You are a fact-checking assistant. You will be given an answer and the
evidence text it was supposed to be based on. Determine whether the
answer's claims are actually supported by the evidence.

Return valid JSON with this shape:
{
  "grounded": true or false,
  "reason": "one short sentence explaining your judgment"
}

Do not output markdown, prose, or extra keys.
""".strip()

    groundedness_user_prompt = f"""
Answer to check:
{result['final_answer']}

Evidence it should be based on:
{result['evidence_text']}
""".strip()

    try:
        groundedness_json = await call_llm_and_parse(groundedness_system_prompt, groundedness_user_prompt, model=LLM_MODEL)
        if groundedness_json.get('grounded') is False:
            print("WARNING: This answer may not be fully supported by the retrieved evidence.")
            print(f"Reason: {groundedness_json.get('reason', 'No reason given.')}")
        else:
            print("PASSED — answer is supported by the retrieved evidence.")
    except Exception:
        print("(Groundedness check could not be completed.)")
else:
    print("(No evidence to check against.)")

### 6f. LLM-as-Judge Comparison

In [ ]:
# --- LLM-as-Judge evaluation ---
# Scores the answer across four criteria (correctness, completeness,# conciseness, faithfulness) using a separate LLM call as an impartial evaluator.judge_system_prompt = """
You are an impartial evaluator ("LLM-as-Judge") for a RAG system.
You will be given a question, the final answer produced by the system,
the evidence it was based on, and the explainability summary.

Score the answer on each criterion below using a 1-5 scale:

| Criterion        | 1 (Poor)                        | 5 (Excellent)                              |
|------------------|----------------------------------|---------------------------------------------|
| Correctness      | Factually wrong                  | Fully correct, no errors                    |
| Completeness     | Misses key parts of the question | Answers all parts of the question            |
| Conciseness      | Verbose or contains filler        | Tight, no unnecessary information           |
| Faithfulness     | Invents facts not in evidence    | Every claim traceable to provided evidence  |

Return valid JSON with this shape:
{
  "correctness": {"score": int, "reason": "short explanation"},
  "completeness": {"score": int, "reason": "short explanation"},
  "conciseness": {"score": int, "reason": "short explanation"},
  "faithfulness": {"score": int, "reason": "short explanation"},
  "overall_score": float,
  "overall_summary": "one-paragraph overall assessment"
}

overall_score should be the average of the four scores, rounded to one decimal.
Do not output markdown, prose, or extra keys.
""".strip()

judge_user_prompt = f"""
Question:
{result['question']}

Final Answer:
{result['final_answer']}

Evidence:
{result['evidence_text']}

Explainability:
{json.dumps(result['explainability'], indent=2)}
""".strip()

try:
    judge_json = await call_llm_and_parse(judge_system_prompt, judge_user_prompt, model=LLM_MODEL)

    print("=" * 70)
    print("LLM-AS-JUDGE EVALUATION")
    print("=" * 70)
    for criterion in ["correctness", "completeness", "conciseness", "faithfulness"]:
        entry = judge_json.get(criterion, {})
        print(f"\n{criterion.upper()}: {entry.get('score', '?')}/5")
        print(f"  {entry.get('reason', '')}")
    print(f"\nOVERALL: {judge_json.get('overall_score', '?')}/5")
    print(f"{judge_json.get('overall_summary', '')}")
except Exception as e:
    print(f"(LLM-as-Judge evaluation could not be completed: {e})")